# Notebook 8  - Room-segmentation evaluation (paper mean IoU + PQ)

**Stage GT file · runs last (after Notebook 7)**

> **Pipeline execution order: `6 -> 1 -> 2 -> 4 -> 7 -> 8`.**
> Notebook 8 scores each segmentation method's room-label map against the Notebook 7 ground
> truth. The **headline metric is the paper's point-based mean room IoU** (Albadri et al., ISPRS
> 2025, Eq. 6/7) with `p_i`/`g_i` matching (Eq. 3/4)  - directly comparable to the published
> baseline. Panoptic Quality (SQ / RQ / PQ) is kept as a **secondary** number.

## Purpose
Per method: **mean room IoU** + over-/under-segmentation counts (primary), and SQ/RQ/PQ
(secondary). All scoring lives in `scan2bim.eval` (pure + unit-tested); this notebook is a thin
driver.

## Inputs
- `stage_gt/gt_room_labels.npy` (Notebook 7 — clean interior-area GT).
- `stage1_occupancy/wall_mask.npy` — source of the **canonical wall scaffold** (cleaned via
  `scan2bim.eval_wall_scaffold`); every method + the GT exclude the *same* wall pixels (Task 04/05).
- `stage1_occupancy/coverage.npy` — the scan-coverage raster, for the harmonized void filter (Task 05).
- Each method reads its **own** stage (no aliasing — Task 03), via `scan2bim.load_method_labels`:
  - **Geometry**       ← `stage2_watershed/room_labels.npy`
  - **SAM**            ← `stage_sam_auto/room_labels.npy`  (pure automatic SAM, Method 2)
  - **Geometry + SAM** ← `stage4_sam_refined/room_labels_refined.npy`  (watershed refined by SAM)

## Outputs  (`{out_root}/stage_gt/`)
- `room_results.json`  - per method: `mean_iou`, `per_room` (per-room IoU), `over_seg`,
  `under_seg`, `matched_pairs`, room counts (Task 07 aggregates this across areas).
- `pq_results.json`  - per method `dict(SQ, RQ, PQ, TP, FP, FN)` (secondary).

## Assumptions
- All label maps share the Stage 1 grid (same `H x W`, same transform), so pixels correspond. A
  method whose stage is absent, or whose grid does not match the GT, is **skipped cleanly** ("not
  run") rather than crashing or being aliased to another method's array.

### Setup

In [1]:
import sys, os, json
import numpy as np

PROJECT_ROOT = os.path.abspath('../..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
sys.path = [p for p in sys.path if not p.endswith('/src') and not p.endswith('\\src')]
for mod in list(sys.modules):
    if mod == 'scan2bim' or mod.startswith('scan2bim.'):
        del sys.modules[mod]

import scan2bim
from scan2bim import artifacts as A

CFG = scan2bim.load_config()
SHOW_DEBUG = True
STAGE_GT = 'stage_gt'
print('scan2bim', scan2bim.__version__, '| output root:', CFG.out_root)

scan2bim 1.0.0 | output root: c:\onestruction\scan2bim_out


### Step 1  - Load the GT, the shared wall scaffold, and each method's own labels
`scan2bim.load_method_labels` reads each method from its **own** stage (never aliasing two
methods). A method whose stage is missing (`None`) or whose grid does not match the GT is dropped
here with a printed reason, so the comparison only scores methods that actually ran on this grid.
The Stage-1 `wall_mask` is the shared scaffold passed to every score.

In [2]:
gt_dir = A.stage_dir(CFG.out_root, STAGE_GT)
gt_labels = A.load_npy(os.path.join(gt_dir, 'gt_room_labels.npy')).astype('int32')

# Canonical wall scaffold (research-fixes Task 05): the CLEANED Stage-1 slab-occupancy wall mask
# (scan2bim.eval_wall_scaffold). GT and every prediction exclude the SAME wall pixels — the one
# scaffold Tasks 02/04 reference too. Used by the SUPPLEMENTARY pixel/PQ scorers; the paper
# protocol below stays scaffold-free (faithful to Albadri).
s1 = A.load_stage_dir(CFG.out_root, A.STAGE1)
wall_mask = A.load_npy(os.path.join(s1, A.WALLMASK_NPY)).astype(bool)
coverage  = A.load_npy(os.path.join(s1, A.COVERAGE_NPY)).astype(bool)
assert wall_mask.shape == gt_labels.shape, (wall_mask.shape, gt_labels.shape)
walls_eval = scan2bim.eval_wall_scaffold(wall_mask, CFG)

# Paper protocol (Task 02b) is POINT-based: it needs the Stage-1 transform + the GT interior
# POINTS (not the rasterised labels). IoU is counted over points, matched at p/g >= 75%.
tf = A.load_transform(os.path.join(s1, A.TRANSFORM_JSON))
gt_points, gt_point_ids = scan2bim.load_gt_room_points(CFG.gt_dir)

# Each method from its OWN stage (no aliasing). None -> stage not run; shape!=GT -> stale stage.
raw = scan2bim.load_method_labels(CFG.out_root)
methods = {}
for name, lab in raw.items():
    if lab is None:
        print('%-16s SKIP  - stage not run' % name)
    elif lab.shape != gt_labels.shape:
        print('%-16s SKIP  - grid %s != GT %s (stale stage; re-run on this grid)'
              % (name, lab.shape, gt_labels.shape))
    else:
        methods[name] = lab
print('GT rooms          :', len([r for r in np.unique(gt_labels) if r >= 1]),
      '| GT interior pts:', len(gt_points), '| wall scaffold px:', int(walls_eval.sum()))
assert methods, 'No method produced labels on the GT grid — run the segmentation notebooks first.'

# HARMONIZED comparison (Task 05): apply ONE area + void + wall-scaffold filter to every method
# so the head-to-head measures the SEGMENTER, not each method's own post-filter. eval_profile
# 'comparison' harmonizes (eval_min_room_area_m2 / eval_min_coverage_frac); 'paper' passes through.
methods_eval = {name: scan2bim.harmonize_room_labels(lab, coverage, walls_eval, CFG)
                for name, lab in methods.items()}
print('
HARMONIZED filter (Task 05) — profile=%r  area>=%.2f m^2  coverage>=%.2f'
      % (CFG.eval_profile, CFG.eval_min_room_area_m2, CFG.eval_min_coverage_frac))
for name in methods:
    n_raw = len([r for r in np.unique(methods[name]) if r >= 1])
    n_h   = len([r for r in np.unique(methods_eval[name]) if r >= 1])
    print('  %-16s rooms %3d -> %3d' % (name, n_raw, n_h))

SyntaxError: unterminated string literal (detected at line 39) (3705696696.py, line 39)

### Step 2 — Scorers: paper protocol (primary) + pixel IoU / PQ (supplementary)
**Primary — `scan2bim.score_rooms_paper`** (Albadri et al. 2025, Eq. 2-7, point-based). Each GT
interior point is labelled by the predicted room of the cell it lands in; `IoU = |P∩G|/|P∪G|`
(Eq. 6) is counted **over points**, and predicted↔GT rooms are matched at `p_i` or `g_i >= 75%`
(Eq. 5). The headline is **mean IoU over *matched* rooms** + **detection rate** (matched / #GT) +
over-/under-seg — directly comparable to the paper's Table 2 + Fig. 5. No wall scaffold (faithful).

**Supplementary (our own, NOT in the paper)** — these run on the **harmonized** labels (Task 05:
one shared area + void + canonical-wall-scaffold filter), so they compare segmenters, not post-filters:
- `scan2bim.score_rooms` — *pixel* mean IoU over **all** GT rooms (missed = 0), i.e. stricter than
  the paper; uses the shared wall scaffold.
- `compute_pq` — Panoptic Quality (`SQ = mean IoU over TPs`, `RQ = TP/(TP + 0.5·FP + 0.5·FN)`,
  `PQ = SQ·RQ`; greedy match at `IoU > 0.5`), also on the wall scaffold.

In [ ]:
# Primary scorer is scan2bim.score_rooms (paper Eq. 3/4/6/7)  - imported, unit-tested, pure.
# Secondary PQ scorer below shares the wall scaffold so both metrics see the same valid cells.
def compute_pq(pred_labels, gt_labels, wall_mask=None):
    valid = np.ones(gt_labels.shape, bool) if wall_mask is None else ~np.asarray(wall_mask, bool)
    pred_ids = np.array([i for i in np.unique(pred_labels) if i >= 1])
    gt_ids = np.array([i for i in np.unique(gt_labels) if i >= 1])
    P, G = len(pred_ids), len(gt_ids)
    if P == 0 or G == 0:
        return dict(SQ=0.0, RQ=0.0, PQ=0.0, TP=0, FP=int(P), FN=int(G))

    pred_area = np.array([(valid & (pred_labels == v)).sum() for v in pred_ids], np.int64)
    gt_area = np.array([(valid & (gt_labels == v)).sum() for v in gt_ids], np.int64)

    inter = np.zeros((P, G), np.int64)
    both = valid & (pred_labels >= 1) & (gt_labels >= 1)
    if both.any():
        # pred_ids / gt_ids are sorted (np.unique), so searchsorted maps a label to its index.
        pr = np.searchsorted(pred_ids, pred_labels[both])
        gr = np.searchsorted(gt_ids, gt_labels[both])
        np.add.at(inter, (pr, gr), 1)

    union = pred_area[:, None] + gt_area[None, :] - inter
    iou = np.where(union > 0, inter / union, 0.0)

    pairs = sorted(((iou[i, j], i, j)
                    for i in range(P) for j in range(G) if iou[i, j] > 0.5),
                   reverse=True)
    pred_taken, gt_taken, matched = set(), set(), []
    for v, i, j in pairs:
        if i in pred_taken or j in gt_taken:
            continue
        pred_taken.add(i); gt_taken.add(j); matched.append(v)

    TP = len(matched); FP = P - TP; FN = G - TP
    SQ = float(np.mean(matched)) if TP else 0.0
    denom = TP + 0.5 * FP + 0.5 * FN
    RQ = TP / denom if denom else 0.0
    return dict(SQ=SQ, RQ=RQ, PQ=SQ * RQ, TP=int(TP), FP=int(FP), FN=int(FN))

### Step 3 — Score every method + print the tables
Primary table: the paper's **mean room IoU** + over-/under-seg (RAW labels, faithful). Secondary
tables: pixel IoU + PQ on the **harmonized** labels + canonical wall scaffold (Task 05).

In [ ]:
# Primary = point-based paper protocol (faithful to Albadri — RAW labels, no harmonization).
# Supplementary pixel IoU + PQ run on the HARMONIZED labels + canonical scaffold (Task 05), so
# they compare segmenters apples-to-apples rather than each method's own post-filter.
paper_results = {name: scan2bim.score_rooms_paper(gt_points, gt_point_ids, lab, tf)
                 for name, lab in methods.items()}
room_results = {name: scan2bim.score_rooms(lab, gt_labels, wall_mask=walls_eval)
                for name, lab in methods_eval.items()}
pq_results = {name: compute_pq(lab, gt_labels, wall_mask=walls_eval)
              for name, lab in methods_eval.items()}

print('PAPER PROTOCOL — Albadri et al. 2025 (point-based IoU, match @ p/g>=75%, Eq. 2-7)')
print('Comparable to their Table 2 + Fig. 5.  meanIoU is over MATCHED rooms only.')
print('Method            #GT  #seg  over  under  detect%   meanIoU(matched)')
print('-' * 68)
for name, r in paper_results.items():
    print('%-16s  %3d  %4d  %4d  %5d   %5.1f%%      %6.1f%%'
          % (name, r['n_gt'], r['n_seg'], r['over_seg'], r['under_seg'],
             100 * r['detection_rate'], 100 * r['mean_iou_matched']))

print('\nSUPPLEMENTARY (our own, NOT the paper) — pixel mean IoU over ALL GT rooms (stricter)')
print('Method            meanIoU   over-seg  under-seg   #pred  #GT')
print('-' * 62)
for name, r in room_results.items():
    print('%-16s  %6.1f%%   %6d    %7d     %4d  %4d'
          % (name, 100 * r['mean_iou'], r['over_seg'], r['under_seg'],
             r['n_pred_rooms'], r['n_gt_rooms']))

print('\nSUPPLEMENTARY (our own) — Panoptic Quality (greedy match @ IoU>0.5)')
print('Method              SQ      RQ      PQ      TP   FP   FN')
print('-' * 58)
for name, r in pq_results.items():
    print('%-16s  %5.1f%%  %5.1f%%  %5.1f%%   %3d  %3d  %3d'
          % (name, 100 * r['SQ'], 100 * r['RQ'], 100 * r['PQ'],
             r['TP'], r['FP'], r['FN']))

NameError: name 'methods' is not defined

### Step 4  - Save `room_results.json` (primary) + `pq_results.json` (secondary)
`room_results.json` carries the full per-room IoU / matched-pairs / over-/under-seg per method so
Task 07 can aggregate across areas.

In [ ]:
out_dir = A.ensure_dir(gt_dir)
paper_path = os.path.join(out_dir, 'room_results_paper.json')   # primary — paper-comparable
room_path = os.path.join(out_dir, 'room_results.json')          # supplementary — pixel, all rooms
pq_path = os.path.join(out_dir, 'pq_results.json')              # supplementary — PQ
with open(paper_path, 'w') as f:
    json.dump(paper_results, f, indent=2)
with open(room_path, 'w') as f:
    json.dump(room_results, f, indent=2)
with open(pq_path, 'w') as f:
    json.dump(pq_results, f, indent=2)
print('packaged ->', paper_path)
print('packaged ->', room_path)
print('packaged ->', pq_path)